<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part B: Statistical Forecasting</h2>
<h2>Notebook B03: Advanced Statistical Methods</h2>
</div>

Exponential smoothing and ARIMA both assume you can name the structure in advance: one season, of one
fixed length, with correlations that die away within a few lags. Plenty of real series break at least one
of those assumptions. Electricity demand has a daily cycle *and* a weekly one. A yearly cycle on daily
data has a period of 365.25, which is not an integer. Retail sales jump on holidays that move around the
calendar.

This notebook covers four methods that address those gaps, and it is as much about knowing when to reach
for them as about the methods themselves. One of them turns out to be the simplest model in the whole
course.

---

**Contents**

1. [Imports and Data Loading](#1.-Imports-and-Data-Loading)
2. [The Theta Method](#2.-The-Theta-Method)
3. [When One Season Is Not Enough](#3.-When-One-Season-Is-Not-Enough)
4. [Dynamic Harmonic Regression](#4.-Dynamic-Harmonic-Regression)
5. [TBATS and BATS](#5.-TBATS-and-BATS)
6. [Prophet](#6.-Prophet)
7. [Choosing Among Them](#7.-Choosing-Among-Them)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-Data-Loading">1. Imports and Data Loading</h3>
</div>

In [ ]:
import importlib.util
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.deterministic import DeterministicProcess, Fourier
from statsmodels.tsa.forecasting.theta import ThetaModel
from statsmodels.tsa.statespace.sarimax import SARIMAX

import nb_config

sns.set_theme(style="whitegrid")

Two datasets. The CDC temperature series carries on from Notebooks
[B01](./B01_Exponential_smoothing_models.ipynb) and [B02](./B02_ARIMA_models.ipynb), so the Theta and
Prophet results can be compared with everything before them. For the multiple-seasonality sections we
switch to hourly electricity load, which has two cycles at once.

Results so far on the temperature series, for reference:

| Method | Test MAE |
|---|---|
| Seasonal naive (A05) | 1.74 °C |
| Holt-Winters (B01) | 1.22 °C |
| SARIMA, searched (B02) | 1.31 °C |

In [ ]:
series = pd.read_parquet(nb_config.CDC_TEMP_PATH)["Brandenburg/Berlin"].asfreq("MS")

TEST_MONTHS = 24
SEASON_LENGTH = 12

train = series.iloc[:-TEST_MONTHS]
test = series.iloc[-TEST_MONTHS:]


def mean_absolute_error(actual, forecast):
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))))


SEASONAL_NAIVE_MAE = 1.74
HOLT_WINTERS_MAE = 1.22
SARIMA_MAE = 1.31

print(f"Train: {len(train)} months, Test: {len(test)} months")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-The-Theta-Method">2. The Theta Method</h3>
</div>

The **Theta method** was the surprise winner of the M3 forecasting competition in 2000, beating far more
elaborate entries. Its reputation has held up: it is still a standard benchmark, and it is close to the
simplest thing in this notebook.

The idea is to modify the *curvature* of the series. Multiply the second differences by a factor
$\theta$ to produce a "theta line": $\theta = 0$ gives a straight regression line through the data, which
is pure long-term trend, while $\theta = 2$ doubles the curvature and exaggerates short-term movement.
Forecast each line separately, then average them back together.

The classical version combines $\theta = 0$ and $\theta = 2$, and it turns out to be equivalent to simple
exponential smoothing with drift. `statsmodels` handles the seasonal adjustment around it for you.

In [ ]:
theta = ThetaModel(train, period=SEASON_LENGTH).fit()
theta_forecast = theta.forecast(TEST_MONTHS)

print(f"Test MAE: {mean_absolute_error(test, theta_forecast):.2f} °C")
print()
print(f"  Seasonal naive (A05):   {SEASONAL_NAIVE_MAE:.2f} °C")
print(f"  Holt-Winters (B01):     {HOLT_WINTERS_MAE:.2f} °C")
print(f"  SARIMA, searched (B02): {SARIMA_MAE:.2f} °C")

1.25 °C, from a method with no orders to choose, no grid to search, and essentially no tuning. That puts
it ahead of the SARIMA model we spent 36 fits searching for in Notebook B02, and within 0.03 °C of
Holt-Winters.

This is why Theta is a benchmark rather than a curiosity. When a new method is proposed, beating Theta is
the minimum bar, and a surprising number of elaborate models fail to clear it. It also costs nothing to
try, which makes it a natural companion to the baselines from Notebook A05.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))

ax.plot(train["2021":], color="steelblue", linewidth=1.2, label="Train")
ax.plot(test, color="black", linewidth=1.8, label="Actual")
ax.plot(theta_forecast, color="seagreen", linewidth=1.5, linestyle="--",
        label=f"Theta ({mean_absolute_error(test, theta_forecast):.2f} °C)")

ax.axvline(test.index[0], color="gray", linestyle="--", linewidth=1.0)
ax.set_title("The Theta method", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

**Exercise.** `ThetaModel` takes a `deseasonalize` argument, on by default. Refit with `deseasonalize=False` and compare. How much of Theta's performance on this series comes from the seasonal adjustment rather than from the theta lines themselves?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-When-One-Season-Is-Not-Enough">3. When One Season Is Not Enough</h3>
</div>

Everything so far has assumed a single seasonal cycle. Hourly electricity load has at least two: demand
follows a daily shape, and weekdays differ from weekends.

SARIMA can only take one seasonal period, and it is the wrong tool even for that one here. A weekly cycle
on hourly data means $m = 168$, which means 168 seasonal states to estimate. That is slow to the point of
impractical, and it spends an enormous number of parameters on a smooth, repeating shape.

Let us look at the data first.

In [ ]:
ops = pd.read_parquet(nb_config.OPS_15M_PATH)

hourly_load = (
    ops[(ops["country"] == "AT") & (ops["measure"] == "actual_entsoe_transparency")]["value"]
    .tz_convert(None)
    .resample("h").mean()
    .dropna()
    .asfreq("h")
)

print(f"{len(hourly_load):,} hours, {hourly_load.index.min().date()} to {hourly_load.index.max().date()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

two_weeks = hourly_load["2018-06-04":"2018-06-17"]
axes[0].plot(two_weeks.index, two_weeks.values, color="steelblue", linewidth=1.0)
axes[0].set_title("Two weeks", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Load (MW)")
axes[0].tick_params(axis="x", rotation=45)

year = hourly_load["2018"]
axes[1].plot(year.groupby(year.index.hour).mean(), color="seagreen", marker="o", markersize=4)
axes[1].set_title("Average by hour of day", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Hour")

axes[2].plot(year.groupby(year.index.dayofweek).mean(), color="darkorange", marker="o", markersize=5)
axes[2].set_title("Average by day of week", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Day (0 = Monday)")

for ax in axes:
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Both cycles are clear: a daily swing of roughly 2,000 MW between the overnight trough and the daytime
plateau, and a weekly pattern where Saturday and especially Sunday sit well below the working week.

We forecast **24 hours ahead**, the horizon that matters commercially for electricity, training on the
preceding eight weeks.

In [ ]:
HORIZON = 24
ORIGIN = pd.Timestamp("2018-06-07")
TRAINING_WEEKS = 8

load_train = hourly_load.loc[ORIGIN - pd.Timedelta(weeks=TRAINING_WEEKS):ORIGIN - pd.Timedelta(hours=1)]
load_test = hourly_load.loc[ORIGIN:ORIGIN + pd.Timedelta(hours=HORIZON - 1)]

print(f"Train: {len(load_train)} hours up to {load_train.index[-1]}")
print(f"Test:  {len(load_test)} hours from {load_test.index[0]}")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Dynamic-Harmonic-Regression">4. Dynamic Harmonic Regression</h3>
</div>

**Dynamic harmonic regression** replaces the seasonal states with **Fourier terms**: pairs of sine and
cosine waves at the seasonal frequency and its harmonics. A handful of them reproduces a smooth repeating
shape of any period, integer or not, and you can stack several periods in the same model. Whatever the
waves do not explain is left to ARIMA errors.

The trade is favourable. A weekly cycle needs 168 seasonal states in SARIMA, or six Fourier terms here.
The cost is that the seasonal shape is **fixed**: it cannot evolve over time the way a Holt-Winters
seasonal component can.

In [ ]:
def harmonic_terms(index, periods_and_orders, horizon=None):
    """Fourier terms for one or more seasonal periods, in sample and out."""
    process = DeterministicProcess(
        index,
        constant=True,
        additional_terms=[Fourier(period, order=order) for period, order in periods_and_orders],
    )
    if horizon is None:
        return process.in_sample()
    return process.in_sample(), process.out_of_sample(horizon)


def fit_dhr(train, periods_and_orders, horizon, order=(2, 0, 1)):
    """Dynamic harmonic regression: Fourier seasonality with ARIMA errors."""
    exog_train, exog_future = harmonic_terms(train.index, periods_and_orders, horizon)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = SARIMAX(train, exog=exog_train, order=order, enforce_stationarity=False).fit(disp=False)

    return model, model.forecast(horizon, exog=exog_future)


daily_only, daily_forecast = fit_dhr(load_train, [(24, 4)], HORIZON)
both, both_forecast = fit_dhr(load_train, [(24, 4), (168, 3)], HORIZON)

print(f"Daily only          {daily_only.model.exog.shape[1]:>2} regressors   "
      f"MAE = {mean_absolute_error(load_test, daily_forecast):6.1f} MW")
print(f"Daily + weekly      {both.model.exog.shape[1]:>2} regressors   "
      f"MAE = {mean_absolute_error(load_test, both_forecast):6.1f} MW")

Adding the weekly cycle cuts the error by around a fifth, for the price of six extra columns. That is the
case for dynamic harmonic regression in one line: a second seasonality is nearly free.

For contrast, here is SARIMA given the daily season only, which is as much as it can practically take.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sarima_daily = SARIMAX(
        load_train, order=(2, 0, 1), seasonal_order=(1, 0, 1, 24), enforce_stationarity=False
    ).fit(disp=False)

sarima_forecast = sarima_daily.forecast(HORIZON)

# The baseline for this kind of data: today will look like yesterday
yesterday = pd.Series(load_train.iloc[-24:].values, index=load_test.index, name="Yesterday")

comparison = pd.DataFrame([
    {"Method": "SARIMA (daily season only)", "MAE": mean_absolute_error(load_test, sarima_forecast)},
    {"Method": "DHR (daily)", "MAE": mean_absolute_error(load_test, daily_forecast)},
    {"Method": "DHR (daily + weekly)", "MAE": mean_absolute_error(load_test, both_forecast)},
    {"Method": "Naive: repeat yesterday", "MAE": mean_absolute_error(load_test, yesterday)},
]).sort_values("MAE").reset_index(drop=True)

comparison.round(1)

Two findings, and the second one matters more.

**The harmonic models beat SARIMA and run far faster.** DHR with both seasons fits in under a second;
SARIMA with a single daily season takes several, and a weekly season would be worse than impractical.

**A naive forecast beats all of them, by almost a factor of five.** This is not a quirk of the chosen day.

In [ ]:
origins = pd.date_range("2018-03-15", periods=6, freq="21D")
rows = []

for origin in origins:
    window = hourly_load.loc[origin - pd.Timedelta(weeks=TRAINING_WEEKS):origin - pd.Timedelta(hours=1)]
    actual = hourly_load.loc[origin:origin + pd.Timedelta(hours=HORIZON - 1)]

    _, forecast = fit_dhr(window, [(24, 4), (168, 3)], HORIZON)

    rows.append({
        "Origin": origin.date(),
        "DHR": mean_absolute_error(actual, forecast),
        "Repeat yesterday": mean_absolute_error(actual, window.iloc[-24:].values),
    })

rolling = pd.DataFrame(rows).set_index("Origin")
print(rolling.round(0).to_string())
print()
print(rolling.mean().round(1).to_string())

The naive forecast wins at every origin. The reason is structural rather than a failure of tuning: the
Fourier terms describe the *average* daily and weekly shape over the training window, so the forecast is
an average day. Yesterday's actual load carries today's level, which for a series driven by weather and
economic activity is worth more than a smooth average profile.

This does not make dynamic harmonic regression a bad method. It makes it the wrong model *on its own* for
this horizon. In practice its Fourier terms would be one component of a larger model, alongside the
recent level and a weather forecast, which is roughly what production load-forecasting systems do.

It is also not the first time in this course that a naive forecast has embarrassed something far more
sophisticated, and it will not be the last. That is the habit worth taking away: compute the baseline
first, every time, especially when the model you are building is obviously better.

**Exercise.** Add the previous day's load at the same hour as an exogenous column alongside the Fourier terms, and refit. Does giving the model the level directly close the gap to the naive forecast?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-TBATS-and-BATS">5. TBATS and BATS</h3>
</div>

**TBATS** is the other standard answer to multiple seasonality. The name is an acronym for what it bundles
together:

- **T**rigonometric seasonality, Fourier terms much like the ones above, but with coefficients that are
  allowed to **evolve over time** rather than staying fixed;
- **B**ox-Cox transformation, chosen automatically, to stabilise variance;
- **A**RMA errors for whatever is left;
- **T**rend, with damping;
- **S**easonal components, several at once, and periods need not be whole numbers.

**BATS** is the same without the trigonometric part, so it uses ordinary seasonal states: faster, and fine
when the periods are short and integer.

The appeal is that it decides all of this itself. The cost is that it searches over a large model space,
which is slow on long series, and that its evolving seasonality is harder to interpret than a fixed set
of Fourier coefficients.

> **A practical note on the Python library.** The `tbats` package is the standard implementation, and at
> the time of writing its most recent release (1.1.3) is incompatible with scikit-learn 1.6 and newer: it
> calls a function whose signature has since changed, and fails immediately. The package has not been
> updated in some years. We therefore do **not** install it for this course, and the cell below is shown
> for reference rather than executed.
>
> If you need TBATS today, the practical options are an environment pinned to an older scikit-learn, the
> `statsforecast` library, which provides a maintained implementation, or R's `forecast` package, where
> the method originated.

In [ ]:
# Not executed: see the note above. This is what the call would look like.
#
#     from tbats import TBATS
#
#     estimator = TBATS(seasonal_periods=[24, 168])   # daily and weekly
#     fitted = estimator.fit(load_train.values)
#     forecast = fitted.forecast(steps=HORIZON)
#
# Everything else, the Box-Cox transformation, the ARMA order and the damping,
# is selected automatically.

print("TBATS is described above but not run; the library is currently unmaintained.")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-Prophet">6. Prophet</h3>
</div>

**Prophet**, from Meta, approaches forecasting from a different direction. Rather than a statistical model
of the error process, it is a **decomposable curve-fitting** model:

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

a trend $g(t)$ that is piecewise linear with automatically detected changepoints, seasonality $s(t)$ as
Fourier terms, and holidays $h(t)$ as explicit event effects. Because it is a regression on time rather
than on lagged values, it tolerates missing data and irregular sampling without complaint.

Its real strength is the holiday handling and the fact that its parameters are interpretable by people
who are not statisticians. Its weakness is the flip side of the same design: it does not model
autocorrelation at all, so it leaves short-range structure on the table.

> Prophet is not part of the default environment, since it is a large dependency that only this notebook
> uses. Install it with `uv sync --group advanced`. The cells below skip cleanly if it is absent.

In [ ]:
PROPHET_AVAILABLE = importlib.util.find_spec("prophet") is not None

if PROPHET_AVAILABLE:
    import logging

    from prophet import Prophet

    # Prophet is talkative; quieten its fitting backend
    logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
    print("Prophet is available.")
else:
    print("Prophet is not installed. Run 'uv sync --group advanced' to follow this section.")

In [ ]:
if PROPHET_AVAILABLE:
    # Prophet expects two columns with fixed names: ds (dates) and y (values)
    prophet_train = pd.DataFrame({"ds": train.index, "y": train.values})

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,   # meaningless on monthly data
        daily_seasonality=False,
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(prophet_train)

    future = pd.DataFrame({"ds": test.index})
    prophet_output = model.predict(future)
    prophet_forecast = pd.Series(prophet_output["yhat"].values, index=test.index)

    print(f"Test MAE: {mean_absolute_error(test, prophet_forecast):.2f} °C")
    print(f"  Holt-Winters (B01):     {HOLT_WINTERS_MAE:.2f} °C")
    print(f"  SARIMA, searched (B02): {SARIMA_MAE:.2f} °C")
    print(f"  Theta (this notebook):  {mean_absolute_error(test, theta_forecast):.2f} °C")

In [ ]:
if PROPHET_AVAILABLE:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))

    axes[0].plot(train["2021":], color="steelblue", linewidth=1.2, label="Train")
    axes[0].plot(test, color="black", linewidth=1.8, label="Actual")
    axes[0].plot(prophet_forecast, color="purple", linewidth=1.5, linestyle="--", label="Prophet")
    axes[0].fill_between(
        test.index,
        prophet_output["yhat_lower"],
        prophet_output["yhat_upper"],
        color="purple", alpha=0.15, label="80% interval",
    )
    axes[0].axvline(test.index[0], color="gray", linestyle="--", linewidth=1.0)
    axes[0].set_title("Prophet, with its uncertainty interval", fontsize=14, fontweight="bold")
    axes[0].set_ylabel("Temperature (°C)")
    axes[0].legend(loc="upper left")

    # The decomposition Prophet fits
    trend_component = model.predict(pd.DataFrame({"ds": train.index[-600:]}))
    axes[1].plot(train.index[-600:], trend_component["trend"], color="crimson", linewidth=1.5)
    axes[1].set_title("The trend component Prophet extracted", fontsize=14, fontweight="bold")
    axes[1].set_xlabel("Date")
    axes[1].set_ylabel("Temperature (°C)")

    for ax in axes:
        ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

Prophet lands at 1.29 °C: competitive with everything else, and neither better nor worse in a way that
would decide anything. On a clean monthly series with one stable season, all these methods are measuring
the same noise floor.

The lower panel shows what Prophet gives you that the others do not: a trend component you can read
directly, with changepoints it found on its own. On this series it has picked up the warming visible in
the rolling means of Notebook A02, as a piecewise-linear path rather than a number buried in a parameter
vector.

That interpretability, plus built-in holiday handling, is the reason Prophet became popular in business
settings. It is not a reason to prefer it on accuracy: published comparisons routinely find it beaten by
simpler statistical methods, and Theta beats it here on a fraction of the compute.

**Exercise.** Prophet's holiday support is its strongest feature, and this temperature series cannot show it. Fit Prophet to the Rossmann daily sales from Notebook A04, passing the `StateHoliday` dates via the `holidays` argument. How much does declaring the holidays improve the forecast?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-Choosing-Among-Them">7. Choosing Among Them</h3>
</div>

| Method | Reach for it when | Watch out for |
|---|---|---|
| **Theta** | Always, as a benchmark | Nothing. It is nearly free |
| **DHR** | Several seasons, or a non-integer period | Seasonal shape is fixed; carries no recent level |
| **TBATS** | Several seasons, and you want it automated | Slow; the Python library is unmaintained |
| **Prophet** | Holidays and events matter, non-experts read the output | Ignores autocorrelation; rarely the most accurate |

The results on the temperature series line up like this:

| Method | Test MAE | Notebook |
|---|---|---|
| Holt-Winters | 1.22 °C | B01 |
| Theta | 1.25 °C | here |
| Prophet | 1.29 °C | here |
| SARIMA, searched | 1.31 °C | B02 |
| Seasonal naive | 1.74 °C | A05 |

Four quite different methods, spread across 0.09 °C. Once a model captures the structure that is actually
in the series, they converge, and the remaining differences sit inside the spread we measured across
forecast origins in Notebook A06. Choosing between them on this table would be choosing on noise.

So choose on the things the table does not show: whether you need to explain the model to someone,
whether you need holidays or several seasons, how much compute you have, and how much the library will
cost you to maintain. Accuracy stopped being the deciding factor three methods ago.

---

Every forecast in Part B so far has been a single number per time step, which quietly pretends we know
exactly what will happen. The last notebook of this part replaces that with a range, and asks how to tell
whether a stated uncertainty is honest:
[B04 - Probabilistic Forecasting](./B04_Probabilistic_forecasting.ipynb).

**Solutions.** Worked answers to the 3 exercises above, with the reasoning behind them, are in
[B03_Advanced_statistical_models_solutions.ipynb](../solutions/B03_Advanced_statistical_models_solutions.ipynb). Try each one yourself first.
